# Superstore Exploratory Data Analysis

This notebook profiles the raw Superstore CSV used by the ETL pipeline, validates the business grain, and creates reusable analytical extracts under `data/processed/`.

Main questions:
- Is the raw file complete enough for a sales warehouse?
- Which years, regions, categories, products, and customers drive revenue and profit?
- Where do discounts, losses, and margin pressure appear?
- Which processed outputs should feed documentation, QA, or quick analysis outside PostgreSQL?

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "superstore.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

RAW_PATH

## Load Raw Data

The source file contains non-UTF-8 characters, so the loader tries the same fallback strategy used by the ETL extractor.

In [ ]:
def read_csv_with_fallback(path: Path) -> pd.DataFrame:
    for encoding in ("utf-8", "utf-8-sig", "cp1252", "latin1"):
        try:
            df = pd.read_csv(path, encoding=encoding)
            print(f"Loaded {len(df):,} rows using {encoding}")
            return df
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError("csv", b"", 0, 1, f"Unable to decode {path}")

raw = read_csv_with_fallback(RAW_PATH)
raw.head()

In [ ]:
raw.info()

## Clean and Standardize

This mirrors the production ETL naming convention so notebook findings can be compared directly with API and warehouse outputs.

In [ ]:
df = raw.copy()
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

for col in ["order_date", "ship_date"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

df["postal_code"] = df["postal_code"].fillna("UNKNOWN").astype(str)
df["ship_days"] = (df["ship_date"] - df["order_date"]).dt.days
df["year"] = df["order_date"].dt.year
df["quarter"] = df["order_date"].dt.quarter
df["month"] = df["order_date"].dt.month
df["month_name"] = df["order_date"].dt.month_name()
df["profit_margin_pct"] = np.where(df["sales"] != 0, df["profit"] / df["sales"] * 100, np.nan)
df["is_loss"] = df["profit"] < 0

df.head()

## Data Quality Checks

In [ ]:
quality_summary = pd.DataFrame({
    "metric": [
        "rows",
        "columns",
        "duplicate_rows",
        "missing_order_id",
        "missing_customer_id",
        "missing_product_id",
        "missing_order_date",
        "missing_ship_date",
        "negative_profit_rows",
        "date_min",
        "date_max",
    ],
    "value": [
        len(df),
        df.shape[1],
        df.duplicated().sum(),
        df["order_id"].isna().sum(),
        df["customer_id"].isna().sum(),
        df["product_id"].isna().sum(),
        df["order_date"].isna().sum(),
        df["ship_date"].isna().sum(),
        df["is_loss"].sum(),
        df["order_date"].min(),
        df["order_date"].max(),
    ],
})

missing_by_column = (
    df.isna().sum().rename("missing_count").to_frame()
    .assign(missing_pct=lambda x: x["missing_count"] / len(df) * 100)
    .query("missing_count > 0")
    .sort_values("missing_count", ascending=False)
)

display(quality_summary)
display(missing_by_column)

## Executive KPIs

In [ ]:
kpis = pd.DataFrame([{
    "total_sales": df["sales"].sum(),
    "total_profit": df["profit"].sum(),
    "profit_margin_pct": df["profit"].sum() / df["sales"].sum() * 100,
    "orders": df["order_id"].nunique(),
    "customers": df["customer_id"].nunique(),
    "products": df["product_id"].nunique(),
    "avg_order_line_sales": df["sales"].mean(),
    "avg_discount": df["discount"].mean(),
    "loss_rate_pct": df["is_loss"].mean() * 100,
}])

kpis.T.rename(columns={0: "value"})

## Time Series Analysis

In [ ]:
monthly_revenue = (
    df.groupby(["year", "quarter", "month", "month_name"], as_index=False)
    .agg(
        total_orders=("order_id", "nunique"),
        total_quantity=("quantity", "sum"),
        total_sales=("sales", "sum"),
        total_profit=("profit", "sum"),
        avg_discount=("discount", "mean"),
    )
    .sort_values(["year", "month"])
)
monthly_revenue["profit_margin_pct"] = monthly_revenue["total_profit"] / monthly_revenue["total_sales"] * 100
monthly_revenue["period"] = monthly_revenue["year"].astype(str) + "-" + monthly_revenue["month"].astype(str).str.zfill(2)

fig = px.line(monthly_revenue, x="period", y=["total_sales", "total_profit"], markers=True, title="Monthly Sales and Profit")
fig.show()

monthly_revenue.tail(12)

In [ ]:
yearly_growth = (
    df.groupby("year", as_index=False)
    .agg(total_sales=("sales", "sum"), total_profit=("profit", "sum"), orders=("order_id", "nunique"))
    .sort_values("year")
)
yearly_growth["previous_sales"] = yearly_growth["total_sales"].shift(1)
yearly_growth["yoy_growth_pct"] = (yearly_growth["total_sales"] - yearly_growth["previous_sales"]) / yearly_growth["previous_sales"] * 100

px.bar(yearly_growth, x="year", y="total_sales", text="yoy_growth_pct", title="Yearly Sales and YoY Growth").show()
yearly_growth

## Segment, Region, and Category Performance

In [ ]:
def summarize_dimension(columns):
    summary = (
        df.groupby(columns, as_index=False)
        .agg(
            total_orders=("order_id", "nunique"),
            total_customers=("customer_id", "nunique"),
            total_quantity=("quantity", "sum"),
            total_sales=("sales", "sum"),
            total_profit=("profit", "sum"),
            avg_discount=("discount", "mean"),
            loss_rate_pct=("is_loss", lambda s: s.mean() * 100),
        )
        .sort_values("total_sales", ascending=False)
    )
    summary["profit_margin_pct"] = summary["total_profit"] / summary["total_sales"] * 100
    return summary

revenue_by_region = summarize_dimension(["region"])
segment_summary = summarize_dimension(["segment"])
category_sales = summarize_dimension(["category", "sub_category"])

display(revenue_by_region)
display(segment_summary)
display(category_sales.head(10))

In [ ]:
px.bar(revenue_by_region, x="region", y="total_sales", color="profit_margin_pct", title="Revenue by Region").show()
px.treemap(category_sales, path=["category", "sub_category"], values="total_sales", color="total_profit", title="Category and Sub-Category Revenue").show()

## Products and Customers

In [ ]:
top_products = (
    df.groupby(["product_id", "product_name", "category", "sub_category"], as_index=False)
    .agg(
        total_orders=("order_id", "nunique"),
        total_quantity=("quantity", "sum"),
        total_sales=("sales", "sum"),
        total_profit=("profit", "sum"),
        avg_discount=("discount", "mean"),
    )
)
top_products["profit_margin_pct"] = top_products["total_profit"] / top_products["total_sales"] * 100
top_products = top_products.sort_values(["total_sales", "total_profit"], ascending=False)

top_customers = (
    df.groupby(["customer_id", "customer_name", "segment", "region"], as_index=False)
    .agg(
        total_orders=("order_id", "nunique"),
        total_sales=("sales", "sum"),
        total_profit=("profit", "sum"),
        avg_discount=("discount", "mean"),
    )
)
top_customers["profit_margin_pct"] = top_customers["total_profit"] / top_customers["total_sales"] * 100
top_customers = top_customers.sort_values(["total_sales", "total_profit"], ascending=False)

display(top_products.head(15))
display(top_customers.head(15))

In [ ]:
px.bar(top_products.head(15).sort_values("total_sales"), x="total_sales", y="product_name", color="category", orientation="h", title="Top 15 Products by Sales").show()
px.bar(top_customers.head(15).sort_values("total_sales"), x="total_sales", y="customer_name", color="segment", orientation="h", title="Top 15 Customers by Sales").show()

## Discount and Loss Analysis

In [ ]:
discount_bins = pd.cut(df["discount"], bins=[-0.01, 0, 0.1, 0.2, 0.4, 0.8], labels=["0", "0-10%", "10-20%", "20-40%", "40%+"])
discount_impact = (
    df.assign(discount_band=discount_bins)
    .groupby("discount_band", observed=False, as_index=False)
    .agg(
        rows=("order_id", "count"),
        total_sales=("sales", "sum"),
        total_profit=("profit", "sum"),
        loss_rate_pct=("is_loss", lambda s: s.mean() * 100),
    )
)
discount_impact["profit_margin_pct"] = discount_impact["total_profit"] / discount_impact["total_sales"] * 100

loss_by_category = summarize_dimension(["category", "sub_category"]).sort_values("loss_rate_pct", ascending=False)

display(discount_impact)
display(loss_by_category.head(10))
px.bar(discount_impact, x="discount_band", y="profit_margin_pct", title="Profit Margin by Discount Band").show()

## Shipping Analysis

In [ ]:
shipping_summary = (
    df.groupby("ship_mode", as_index=False)
    .agg(
        orders=("order_id", "nunique"),
        avg_ship_days=("ship_days", "mean"),
        total_sales=("sales", "sum"),
        total_profit=("profit", "sum"),
    )
    .sort_values("orders", ascending=False)
)
shipping_summary["profit_margin_pct"] = shipping_summary["total_profit"] / shipping_summary["total_sales"] * 100
shipping_summary

## Save Processed Outputs

`data/processed/` should not be empty in this project. The raw file stays immutable under `data/raw/`, while `data/processed/` holds reproducible, analysis-ready extracts that can be regenerated from this notebook or from a future batch job.

In [ ]:
outputs = {
    "clean_superstore.csv": df,
    "data_quality_summary.csv": quality_summary,
    "monthly_revenue.csv": monthly_revenue.drop(columns=["period"]),
    "yearly_growth.csv": yearly_growth,
    "revenue_by_region.csv": revenue_by_region,
    "segment_summary.csv": segment_summary,
    "category_sales.csv": category_sales,
    "top_products.csv": top_products,
    "top_customers.csv": top_customers,
    "discount_impact.csv": discount_impact,
    "shipping_summary.csv": shipping_summary,
}

for filename, frame in outputs.items():
    frame.to_csv(PROCESSED_DIR / filename, index=False)

sorted(path.name for path in PROCESSED_DIR.glob("*.csv"))

## Summary of Findings

- The dataset is suitable for a dimensional sales warehouse after column normalization and date parsing.
- Sales and profit should be monitored separately; high revenue does not always imply high margin.
- Discounting has a visible relationship with loss rate and margin compression.
- Region, category, product, and customer cuts are all useful dashboard dimensions.
- The processed extracts created above provide lightweight QA artifacts and quick analysis inputs without querying PostgreSQL.